In [15]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import sys; sys.path.append("..")
%load_ext autoreload
%autoreload 2
import gc, os, time, json
import pandas as pd, lightgbm as lgb, psutil
from src.blocking import build_index, prepare_pool

H, M = "../dataset/hold/", "../models/dev/"
s1_hold = pd.read_csv(H + "s1_hold.tsv", sep="\t")
gt_hold = pd.read_csv(H + "gt_hold.tsv", sep="\t")
pool_df = pd.concat([pd.read_csv(H + "s2_hold.tsv", sep="\t"),
                     pd.read_csv(H + "s3_hold.tsv", sep="\t")], ignore_index=True)

# WHY truth as sets: same format the scorer already uses
truth = {s: (set(m.split(",")) if isinstance(m, str) else set())
         for s, m in zip(gt_hold["source1_entity_id"], gt_hold["matched_entity_ids"])}

proc = psutil.Process(os.getpid())
gb = lambda b: round(b / 1e9, 2)

# WHY gc first: so the "before" number is not inflated by garbage
gc.collect()
r0 = proc.memory_info().rss
t0 = time.time()
index = build_index(pool_df)
r1 = proc.memory_info().rss
prepared = prepare_pool(pool_df)
r2 = proc.memory_info().rss

print("pool rows:", len(pool_df), "(expect 1,402,291) | keys:", len(index))
print("seconds:", round(time.time() - t0, 1))
print("RAM before / after index / after prepared (GB):", gb(r0), gb(r1), gb(r2))
print("free RAM now (GB):", gb(psutil.virtual_memory().available))
# WHY this scale: the full pool is 10,320,219 records
print("rough full-scale index + prepared (GB):", round((r2 - r0) * 10_320_219 / len(pool_df) / 1e9, 1))

pool rows: 1402291 (expect 1,402,291) | keys: 5240781
seconds: 74.4
RAM before / after index / after prepared (GB): 0.95 2.08 3.87
free RAM now (GB): 3.62
rough full-scale index + prepared (GB): 21.5


In [2]:
import lightgbm as lgb, json
from src.pipeline import run_chunked

# WHY the saved config: the features and threshold are frozen, never retuned on this set
cfg = json.load(open(M + "matcher_dev_v4_config.json"))
booster = lgb.Booster(model_file=M + "matcher_dev_v4_lgbm.txt")

t0 = time.time()
pred = run_chunked(s1_hold, index, prepared, booster, cfg, chunk=20_000)
print("total minutes:", round((time.time() - t0) / 60, 1))

# WHY pickle: the run is long, so keep the result and score it again without re-running
pd.to_pickle(pred, H + "pred_hold_v4.pkl")

entities 0-20,000 | 38s | kept rows so far 66,129
entities 20,000-40,000 | 38s | kept rows so far 132,304
entities 40,000-60,000 | 38s | kept rows so far 198,496
entities 60,000-80,000 | 38s | kept rows so far 264,398
entities 80,000-100,000 | 37s | kept rows so far 330,171
entities 100,000-120,000 | 38s | kept rows so far 396,269
entities 120,000-140,000 | 39s | kept rows so far 462,320
entities 140,000-160,000 | 38s | kept rows so far 528,936
entities 160,000-180,000 | 37s | kept rows so far 594,803
entities 180,000-200,000 | 37s | kept rows so far 661,275
entities 200,000-220,000 | 37s | kept rows so far 727,482
entities 220,000-240,000 | 37s | kept rows so far 793,952
entities 240,000-260,000 | 37s | kept rows so far 860,195
entities 260,000-280,000 | 37s | kept rows so far 926,205
entities 280,000-300,000 | 37s | kept rows so far 992,410
total minutes: 9.5


In [3]:
from src.matcher import macro_f05

s1_ids = list(s1_hold["entity_id"])
print("macro F0.5:", round(macro_f05(pred, truth, s1_ids), 4), "(dev validation: 0.9608)")

# WHY pooled precision and recall: same definition as Day 4, so the numbers are comparable
tp = n_pred = n_true = 0
single = single_false = 0
for e in s1_ids:
    p, t = pred.get(e, set()), truth[e]
    if not t:
        single += 1
        single_false += bool(p)
        continue
    tp += len(p & t); n_pred += len(p); n_true += len(t)
print("precision:", round(tp / n_pred, 4), "| recall:", round(tp / n_true, 4))
print("singletons falsely matched:", single_false, "of", single)

# WHY by country: the open item from Day 5, and France will behave differently
country = dict(zip(s1_hold["entity_id"], s1_hold["country"]))
for c in ["US", "India"]:
    ids = [e for e in s1_ids if country[e] == c]
    print(c, round(macro_f05(pred, truth, ids), 4))

macro F0.5: 0.9459 (dev validation: 0.9608)
precision: 0.9764 | recall: 0.9165
singletons falsely matched: 2263 of 16747
US 0.9566
India 0.9298
